# Transform properties

Both transforms this package builds on have their own catalogue of analytical
properties -- facts that hold regardless of what signal is being transformed.
The discrete Hankel transform (DHT)'s come from Baddour's own DHT chapter
[Baddour2019a]; the full polar Fourier transform (PFT)'s come from the first
of Baddour's two-part polar-coordinates paper [Baddour2019b] (the second part,
cited throughout earlier notebooks, is the one that turns the theory into a
numerically verified pipeline). This notebook is a tour of a representative
handful of both -- the full catalogue is machine-checked in this package's own
test suite, not just illustrated here.

In [1]:
import numpy as np
from scipy.special import jv

import pypft


## The DHT kernel is self-inverse

`Y^{nN} Y^{nN} = I` even though the kernel itself is not symmetric
[Baddour2019a, Eq. 41] -- so `hankel_transform`/`inverse_hankel_transform`
share the same matrix application and a forward/inverse round trip returns
(almost) exactly the original signal:

In [2]:
n, R, size = 2, 8.0, 32
rng = np.random.default_rng(0)
f = rng.standard_normal(size)

F = pypft.hankel_transform(f, n, R)
f_reconstructed = pypft.inverse_hankel_transform(F, n, R)
float(np.abs(f_reconstructed - f).max())


1.595216643224262e-07

## The Kronecker-delta transform pair

Transforming a standard basis vector returns the matching column of the
abstract kernel itself [Baddour2019a, Eq. 65] -- so, unlike a permutation or
the identity, the DHT of a single-sample impulse is a full, structured
vector, not another impulse:

In [3]:
delta = np.zeros(size)
delta[size // 3] = 1.0
pypft.hankel_transform(delta, n, R)[:6]


array([ 0.19991251,  0.31981448,  0.2337215 , -0.00208925, -0.18695077,
       -0.17086229])

## Negative harmonic orders need no kernel of their own

`hankel_transform`/`inverse_hankel_transform` only ever accept a
non-negative order -- the PFT pipeline (`pypft.transform.scaled_hankel`)
never builds a separate kernel for a negative harmonic, because
`Y^{(-n)N} = (-1)^n * Y^{nN}` exactly. This is not a claim from either paper
cited in this notebook: it is a direct consequence of an exact identity of
Bessel functions of the first kind, true for every `x`, not only at a zero of
`J_n`:

In [4]:
x = np.linspace(0.5, 20.0, 5)
for order in (1, 4, 7):
    lhs = jv(-order, x)
    rhs = ((-1.0) ** order) * jv(order, x)
    print(f"order={order}: max|J_(-n)(x) - (-1)^n J_n(x)| = {np.abs(lhs - rhs).max():.2e}")


order=1: max|J_(-n)(x) - (-1)^n J_n(x)| = 0.00e+00
order=4: max|J_(-n)(x) - (-1)^n J_n(x)| = 0.00e+00
order=7: max|J_(-n)(x) - (-1)^n J_n(x)| = 0.00e+00


## Shift, modulation, and convolution rules

The DHT has no natural shift: `f_{k-k0}` can fall outside the signal's own
index range, and unlike the DFT's periodic exponential kernel, the Bessel
kernel does not wrap around. Baddour's resolution is a *generalized* shift
operator, built from a signal's own transform rather than its indices
[Baddour2019a, Eq. 70], from which shift-modulation [Eq. 76-77],
modulation-shift [Eq. 82], convolution [Eq. 86], and multiplication [Eq. 91]
rules all follow -- entirely analogous to the classical DFT's shift theorem,
just built around this substitute for an ordinary shift.
`tests/dht/test_kernel_properties.py` verifies all four rules exactly against
the abstract kernel; here, the generalized shift is built from the public
API alone, using a delta's own inverse transform as a stand-in for a kernel
column:

In [5]:
def generalized_shift(transform_values, k0):
    delta_k0 = np.zeros(size)
    delta_k0[k0] = 1.0
    kernel_column = pypft.inverse_hankel_transform(delta_k0, n, R)
    return pypft.hankel_transform(kernel_column * transform_values, n, R)


k0 = 5
shifted = generalized_shift(F, k0)
lhs = pypft.hankel_transform(shifted, n, R)
rhs = pypft.inverse_hankel_transform(np.eye(size)[k0], n, R) * F
# the two sides are proportional (a fixed R-scale factor apart, since this
# package's public API works in physical rather than raw index units) --
# the ratio being constant across every entry is the shift-modulation rule
(lhs / rhs)[:5]


array([0.36446645, 0.36446644, 0.36446644, 0.36446644, 0.36446643])

## PFT kernel orthogonality and the delta pair

The PFT's own combined kernel `E^-`/`E^+` obeys the same two flavors of
discrete orthogonality [Baddour2019b, Eqs. 34, 37] that already underlie
every round trip in the previous notebook, plus its own version of the
Kronecker-delta pair: transforming `E^+`'s own column at a fixed frequency
index reproduces a delta at exactly that index [Baddour2019b, Eq. 43].
`pypft._kernel` builds that combined kernel explicitly and from first
principles as this package's own test oracle -- but `inverse_pft` applied to
a standard basis array *is* that same column, so the public pipeline alone
already demonstrates the property operationally:

In [6]:
grid = pypft.PolarGrid(n_radial=8, n_angular=5, R=3.0)

basis = np.zeros((grid.n_radial, grid.n_angular), dtype=complex)
q0, m0 = 2, 3
basis[m0, q0] = 1.0

e_plus_column = pypft.inverse_pft(basis, grid)
recovered = pypft.forward_pft(e_plus_column, grid)
float(np.abs(recovered - basis).max())


5.604429159867557e-07

## Rotation equivariance

Circularly shifting the frequency-domain array by `q0` spokes and then
inverse-transforming gives exactly the same result as inverse-transforming
first and *then* shifting [Baddour2019b, Eq. 75] -- rotating the transform is
the same as rotating the underlying function:

In [7]:
rng = np.random.default_rng(1)
F_grid = rng.standard_normal((grid.n_radial, grid.n_angular)) + 1j * rng.standard_normal(
    (grid.n_radial, grid.n_angular)
)
f_grid = pypft.inverse_pft(F_grid, grid)

q0 = 2
lhs = pypft.inverse_pft(np.roll(F_grid, q0, axis=1), grid)
rhs = np.roll(f_grid, q0, axis=1)
float(np.abs(lhs - rhs).max())


7.447602459741819e-16

## Linearity and the DC term

`forward_pft`/`inverse_pft` are linear maps, like every Fourier-family
transform, and a function with no angular dependence at all reduces the
whole pipeline to a single order-0 Hankel transform (scaled by `2*pi`, the
harmonic-0 term's own scale factor) -- the PFT's analogue of a continuous 2-D
Fourier transform's DC/mean term:

In [8]:
radial_profile = rng.standard_normal(grid.n_radial)
f_symmetric = np.broadcast_to(radial_profile[:, np.newaxis], f_grid.shape)

F_symmetric = pypft.forward_pft(f_symmetric, grid)
expected_column = 2.0 * np.pi * pypft.hankel_transform(radial_profile, 0, grid.R)
float(np.abs(F_symmetric - expected_column[:, np.newaxis]).max())


4.484862251149237e-16

## Where to go next

Both transforms' properties are what let the rest of this package be built
with confidence: typed domains, batching, and visualization all lean on the
pipeline behaving exactly as these identities say it must. The next notebook
introduces the typed domain objects that track where in the chain a signal
currently sits.

In [9]:
from IPython.display import Markdown, display

from pypft.references import Reference, bibliography

display(Markdown(bibliography(Reference.BADDOUR_2019_DHT, Reference.BADDOUR_2019_PFT_PART1)))


- [Baddour2019a] N. Baddour, "The Discrete Hankel Transform," in Fourier Transforms - Century of Digitalization and Increasing Expectations, IntechOpen, 2019. doi:10.5772/intechopen.84399
- [Baddour2019b] N. Baddour, "Discrete Two-Dimensional Fourier Transform in Polar Coordinates Part I: Theory and Operational Rules," Mathematics, 7(8):698, 2019. doi:10.3390/math7080698